# Decentralized Hiking AI
## 02 - Knuckles Trail Preprocessing and Feature Engineering

**Research Component:** Context-Aware Offline Resource Recommendation

### Objective

This notebook preprocesses the real Knuckles trail and terrain datasets
to construct a reliable trail-level feature table for machine-learning
development.

The preprocessing pipeline will:

- load the 559 OpenStreetMap candidate trail/path geometries;
- preserve the original OSM identifiers and relevant trail attributes;
- remove or exclude unsuitable records only when supported by evidence;
- integrate Copernicus DEM-derived elevation information;
- derive terrain characteristics from point-level elevation profiles;
- calculate representative geographic locations for each candidate path;
- derive emergency/support facility proximity features;
- derive communication-infrastructure proximity features as connectivity proxies;
- prepare the trail-level data for later integration with historical weather context.

### Data Integrity Policy

Only real collected data and transparently derived features are used.

No synthetic or mock observations are introduced.

Missing OpenStreetMap attributes will not be replaced with fabricated values.

Communication structures are treated only as geographic connectivity/remoteness
proxies and are not interpreted as verified mobile network coverage.

### Expected Output

The final output of this notebook will be a clean trail-level feature dataset
containing variables such as:

- trail identifier;
- path type;
- trail length;
- representative latitude and longitude;
- minimum and maximum elevation;
- elevation range;
- elevation gain and loss;
- slope characteristics;
- proximity to emergency/support facilities;
- proximity to mapped communication infrastructure.

Historical weather features and recommendation target labels will be handled
in later stages of the research pipeline.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

current = Path.cwd().resolve()

if current.name == "notebooks":
    PROJECT_ROOT = current.parent.parent
elif (current / "research").exists():
    PROJECT_ROOT = current
else:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

# ------------------------------------------------------------
# Research directories
# ------------------------------------------------------------

RESEARCH_DIR = PROJECT_ROOT / "research"
DATASET_DIR = RESEARCH_DIR / "dataset"

RAW_DIR = DATASET_DIR / "raw"
PROCESSED_DIR = DATASET_DIR / "processed"

RESULTS_DIR = RESEARCH_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verify setup
# ------------------------------------------------------------

print("Environment initialized successfully.")
print()
print("Project root :", PROJECT_ROOT)
print("Dataset      :", DATASET_DIR)
print("Raw data     :", RAW_DIR)
print("Processed    :", PROCESSED_DIR)
print("Results      :", RESULTS_DIR)

print()
print("Dataset exists:", DATASET_DIR.exists())

Environment initialized successfully.

Project root : /Users/mirshad/Desktop/Decentralized-Hiking-AI
Dataset      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/dataset
Raw data     : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/dataset/raw
Processed    : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/dataset/processed
Results      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/results

Dataset exists: True


In [2]:
trail_file = (
    PROCESSED_DIR
    / "trails"
    / "knuckles_candidate_paths.geojson"
)

print("Trail file exists:", trail_file.exists())

if not trail_file.exists():
    raise FileNotFoundError(
        f"Trail dataset not found: {trail_file}"
    )

trails = gpd.read_file(trail_file)

print("\nCandidate paths loaded successfully.")
print("Rows:", len(trails))
print("Columns:", len(trails.columns))
print("CRS:", trails.crs)

print("\nGeometry types:")
print(trails.geometry.geom_type.value_counts())

trails.head()

Trail file exists: True

Candidate paths loaded successfully.
Rows: 559
Columns: 38
CRS: EPSG:4326

Geometry types:
LineString    559
Name: count, dtype: int64


,id,osm_type,osm_id,tags,access,bicycle,bridge,crossing,crossing:markings,cutting,...,sac_scale,smoothness,surface,tracktype,wheelchair,width,verified_hiking_trail,length_km,hiking_relation_ids,geometry
0,way/100146710,way,100146710,"{ ""highway"": ""footway"" }",None,None,None,None,None,None,...,None,None,None,None,None,None,False,1.654853,[ ],"LINESTRING (80.71194 7.41703, 80.71188 7.4173,..."
1,way/100146711,way,100146711,"{ ""highway"": ""track"", ""maxspeed"": ""25"", ""motor...",None,None,None,None,None,None,...,None,None,None,None,None,None,False,4.966901,[ ],"LINESTRING (80.69633 7.42838, 80.6967 7.42826,..."
2,way/100146901,way,100146901,"{ ""highway"": ""path"", ""note"": ""Only for Mountai...",None,None,None,None,None,None,...,None,None,None,None,None,None,False,0.859729,[ ],"LINESTRING (80.71186 7.43304, 80.71182 7.43317..."
3,way/100148512,way,100148512,"{ ""highway"": ""track"", ""maxspeed"": ""10"" }",None,None,None,None,None,None,...,None,None,None,None,None,None,False,0.995269,[ ],"LINESTRING (80.71059 7.42809, 80.71073 7.4281,..."
4,way/100149325,way,100149325,"{ ""highway"": ""footway"", ""note"": ""Only for Moun...",None,None,None,None,None,None,...,None,None,None,None,None,None,False,0.646777,[ ],"LINESTRING (80.71048 7.44174, 80.71058 7.44228..."


## 1. Candidate Trail Attribute Assessment

The OpenStreetMap dataset contains candidate paths rather than a fully
verified set of hiking trails.

Before preprocessing, the available OSM attributes are examined to determine
which variables are sufficiently populated and meaningful for the research.

No path is removed solely because optional OpenStreetMap attributes are
missing. Exclusion decisions will be based on explicit evidence such as
access restrictions or tags indicating that a path is unsuitable for hiking.

In [3]:
print("AVAILABLE TRAIL COLUMNS")
print("=" * 70)

for i, column in enumerate(trails.columns, start=1):
    print(f"{i:2}. {column}")

AVAILABLE TRAIL COLUMNS
 1. id
 2. osm_type
 3. osm_id
 4. tags
 5. access
 6. bicycle
 7. bridge
 8. crossing
 9. crossing:markings
10. cutting
11. description
12. embankment
13. foot
14. footway
15. highway
16. horse
17. incline
18. informal
19. lanes
20. layer
21. lit
22. maxspeed
23. motor_vehicle
24. motorcycle
25. mtb:scale
26. mtb:scale:uphill
27. name
28. note
29. sac_scale
30. smoothness
31. surface
32. tracktype
33. wheelchair
34. width
35. verified_hiking_trail
36. length_km
37. hiking_relation_ids
38. geometry


In [4]:
trail_missing = pd.DataFrame({
    "missing_count": trails.isna().sum(),
    "missing_percent": (
        trails.isna().mean() * 100
    ).round(2)
})

trail_missing = trail_missing.sort_values(
    "missing_percent",
    ascending=False
)

trail_missing

,missing_count,missing_percent
cutting,558,99.82
footway,558,99.82
sac_scale,558,99.82
mtb:scale,558,99.82
crossing,558,99.82
crossing:markings,558,99.82
lit,558,99.82
description,558,99.82
embankment,558,99.82
mtb:scale:uphill,558,99.82


In [5]:
important_trail_columns = [
    "highway",
    "access",
    "foot",
    "bicycle",
    "motor_vehicle",
    "surface",
    "tracktype",
    "smoothness",
    "sac_scale",
    "verified_hiking_trail"
]

for column in important_trail_columns:
    if column in trails.columns:
        print("\n" + "=" * 60)
        print(column.upper())
        print("=" * 60)

        print(
            trails[column]
            .value_counts(dropna=False)
            .head(30)
        )


HIGHWAY
highway
track        289
footway      125
path         124
steps         20
bridleway      1
Name: count, dtype: int64

ACCESS
access
None          551
no              4
permissive      3
permit          1
Name: count, dtype: int64

FOOT
foot
None      545
yes        12
permit      2
Name: count, dtype: int64

BICYCLE
bicycle
None    544
no       10
yes       5
Name: count, dtype: int64

MOTOR_VEHICLE
motor_vehicle
None    552
no        5
yes       2
Name: count, dtype: int64

SURFACE
surface
None               514
ground              15
concrete            11
unpaved              6
gravel               4
dirt                 2
paved                2
concrete:lanes       1
asphalt              1
rock                 1
mud                  1
concrete:plates      1
Name: count, dtype: int64

TRACKTYPE
tracktype
None      538
grade3      8
grade2      6
grade4      5
grade1      2
Name: count, dtype: int64

SMOOTHNESS
smoothness
None            552
very_bad          3
impassable 

In [6]:
# Convert OSM tag dictionaries to searchable text
tag_text = trails["tags"].astype(str).str.lower()

restriction_keywords = [
    "mountainbike",
    "mountain bike",
    "no pedestrians",
    "no pedestrian",
    "private",
    "foot=no",
    '"foot": "no"',
    "access=no",
    '"access": "no"'
]

potentially_unsuitable = pd.Series(
    False,
    index=trails.index
)

for keyword in restriction_keywords:
    potentially_unsuitable |= tag_text.str.contains(
        keyword,
        regex=False,
        na=False
    )

potentially_unsuitable_paths = trails.loc[
    potentially_unsuitable,
    [
        col for col in [
            "id",
            "highway",
            "name",
            "access",
            "foot",
            "bicycle",
            "length_km",
            "tags"
        ]
        if col in trails.columns
    ]
].copy()

print(
    "Potentially unsuitable/restricted records:",
    len(potentially_unsuitable_paths)
)

potentially_unsuitable_paths

Potentially unsuitable/restricted records: 6


,id,highway,name,access,foot,bicycle,length_km,tags
2,way/100146901,path,None,None,None,None,0.859729,"{ ""highway"": ""path"", ""note"": ""Only for Mountai..."
4,way/100149325,footway,None,None,None,None,0.646777,"{ ""highway"": ""footway"", ""note"": ""Only for Moun..."
31,way/249336982,path,None,no,yes,no,0.877428,"{ ""access"": ""no"", ""bicycle"": ""no"", ""foot"": ""ye..."
308,way/776284374,path,None,no,yes,None,0.972473,"{ ""access"": ""no"", ""foot"": ""yes"", ""highway"": ""p..."
343,way/835238015,footway,Dawatagala - Knuckles 5 Peaks Nature Trail,no,permit,None,3.738512,"{ ""access"": ""no"", ""description"": ""This is a hi..."
502,way/1246011916,footway,None,no,yes,None,0.598044,"{ ""access"": ""no"", ""foot"": ""yes"", ""highway"": ""f..."


In [7]:
pd.set_option("display.max_colwidth", None)

columns_to_show = [
    col for col in [
        "id",
        "highway",
        "name",
        "access",
        "foot",
        "bicycle",
        "motor_vehicle",
        "motorcycle",
        "note",
        "description",
        "verified_hiking_trail",
        "length_km",
        "tags"
    ]
    if col in trails.columns
]

potentially_unsuitable_paths[
    [
        col for col in columns_to_show
        if col in potentially_unsuitable_paths.columns
    ]
]

,id,highway,name,access,foot,bicycle,length_km,tags
2,way/100146901,path,None,None,None,None,0.859729,"{ ""highway"": ""path"", ""note"": ""Only for Mountainbikes"" }"
4,way/100149325,footway,None,None,None,None,0.646777,"{ ""highway"": ""footway"", ""note"": ""Only for Mountainbikes"" }"
31,way/249336982,path,None,no,yes,no,0.877428,"{ ""access"": ""no"", ""bicycle"": ""no"", ""foot"": ""yes"", ""highway"": ""path"", ""horse"": ""no"", ""informal"": ""yes"", ""maxspeed"": ""25"", ""motor_vehicle"": ""no"", ""motorcycle"": ""yes"" }"
308,way/776284374,path,None,no,yes,None,0.972473,"{ ""access"": ""no"", ""foot"": ""yes"", ""highway"": ""path"", ""surface"": ""unpaved"" }"
343,way/835238015,footway,Dawatagala - Knuckles 5 Peaks Nature Trail,no,permit,None,3.738512,"{ ""access"": ""no"", ""description"": ""This is a hike of the 5 peaks that give the Knuckles Mountain Range their name. This is in the Knuckles Forest Conservation Area, and a permit from the Forest Department is required. The trail head shown as the ending location of the gps track is reachabl"", ""foot"": ""permit"", ""highway"": ""footway"", ""name"": ""Dawatagala - Knuckles 5 Peaks Nature Trail"", ""surface"": ""ground"", ""width"": ""0.5"" }"
502,way/1246011916,footway,None,no,yes,None,0.598044,"{ ""access"": ""no"", ""foot"": ""yes"", ""highway"": ""footway"" }"


In [8]:
flagged_indices = potentially_unsuitable_paths.index

trails.loc[
    flagged_indices,
    columns_to_show
]

,id,highway,name,access,foot,bicycle,motor_vehicle,motorcycle,note,description,verified_hiking_trail,length_km,tags
2,way/100146901,path,None,None,None,None,None,None,Only for Mountainbikes,None,False,0.859729,"{ ""highway"": ""path"", ""note"": ""Only for Mountainbikes"" }"
4,way/100149325,footway,None,None,None,None,None,None,Only for Mountainbikes,None,False,0.646777,"{ ""highway"": ""footway"", ""note"": ""Only for Mountainbikes"" }"
31,way/249336982,path,None,no,yes,no,no,yes,None,None,False,0.877428,"{ ""access"": ""no"", ""bicycle"": ""no"", ""foot"": ""yes"", ""highway"": ""path"", ""horse"": ""no"", ""informal"": ""yes"", ""maxspeed"": ""25"", ""motor_vehicle"": ""no"", ""motorcycle"": ""yes"" }"
308,way/776284374,path,None,no,yes,None,None,None,None,None,False,0.972473,"{ ""access"": ""no"", ""foot"": ""yes"", ""highway"": ""path"", ""surface"": ""unpaved"" }"
343,way/835238015,footway,Dawatagala - Knuckles 5 Peaks Nature Trail,no,permit,None,None,None,None,"This is a hike of the 5 peaks that give the Knuckles Mountain Range their name. This is in the Knuckles Forest Conservation Area, and a permit from the Forest Department is required. The trail head shown as the ending location of the gps track is reachabl",False,3.738512,"{ ""access"": ""no"", ""description"": ""This is a hike of the 5 peaks that give the Knuckles Mountain Range their name. This is in the Knuckles Forest Conservation Area, and a permit from the Forest Department is required. The trail head shown as the ending location of the gps track is reachabl"", ""foot"": ""permit"", ""highway"": ""footway"", ""name"": ""Dawatagala - Knuckles 5 Peaks Nature Trail"", ""surface"": ""ground"", ""width"": ""0.5"" }"
502,way/1246011916,footway,None,no,yes,None,None,None,None,None,False,0.598044,"{ ""access"": ""no"", ""foot"": ""yes"", ""highway"": ""footway"" }"


In [9]:
# Work on a copy so the original imported dataset remains unchanged.
trail_work = trails.copy()

# ------------------------------------------------------------
# Explicit pedestrian prohibition
# ------------------------------------------------------------

trail_work["pedestrian_prohibited"] = (
    trail_work["foot"]
    .fillna("")
    .astype(str)
    .str.lower()
    .eq("no")
)

# ------------------------------------------------------------
# Hiking may require permission
# ------------------------------------------------------------

trail_work["pedestrian_permit_required"] = (
    trail_work["foot"]
    .fillna("")
    .astype(str)
    .str.lower()
    .isin(["permit", "private"])
)

# ------------------------------------------------------------
# General OSM access restriction
# ------------------------------------------------------------

trail_work["general_access_restricted"] = (
    trail_work["access"]
    .fillna("")
    .astype(str)
    .str.lower()
    .isin(["no", "private"])
)

print("Total candidate paths:", len(trail_work))

print(
    "Explicit foot=no:",
    trail_work["pedestrian_prohibited"].sum()
)

print(
    "Foot permit/private:",
    trail_work["pedestrian_permit_required"].sum()
)

print(
    "General access restricted:",
    trail_work["general_access_restricted"].sum()
)

Total candidate paths: 559
Explicit foot=no: 0
Foot permit/private: 2
General access restricted: 4


In [11]:
text_columns = [
    col for col in ["note", "description"]
    if col in trail_work.columns
]

combined_notes = pd.Series(
    "",
    index=trail_work.index,
    dtype="object"
)

for col in text_columns:
    combined_notes = (
        combined_notes
        + " "
        + trail_work[col].fillna("").astype(str)
    )

combined_notes = combined_notes.str.lower()

trail_work["explicit_non_hiking_note"] = (
    combined_notes.str.contains(
        "only for mountainbike",
        regex=False
    )
    |
    combined_notes.str.contains(
        "only for mountain bike",
        regex=False
    )
)

print(
    "Paths explicitly described as non-hiking:",
    trail_work["explicit_non_hiking_note"].sum()
)

trail_work.loc[
    trail_work["explicit_non_hiking_note"],
    [
        "id",
        "highway",
        "name",
        "foot",
        "access",
        "note",
        "length_km"
    ]
]

Paths explicitly described as non-hiking: 2


,id,highway,name,foot,access,note,length_km
2,way/100146901,path,None,None,None,Only for Mountainbikes,0.859729
4,way/100149325,footway,None,None,None,Only for Mountainbikes,0.646777


In [12]:
trail_work["exclude_from_hiking_candidates"] = (
    trail_work["pedestrian_prohibited"]
    |
    trail_work["explicit_non_hiking_note"]
)

excluded_paths = trail_work.loc[
    trail_work["exclude_from_hiking_candidates"]
].copy()

retained_paths = trail_work.loc[
    ~trail_work["exclude_from_hiking_candidates"]
].copy()

print("Original candidate paths :", len(trail_work))
print("Excluded paths           :", len(excluded_paths))
print("Retained candidate paths :", len(retained_paths))

print("\nExclusion reasons:")

print(
    "foot=no:",
    trail_work["pedestrian_prohibited"].sum()
)

print(
    "explicit non-hiking note:",
    trail_work["explicit_non_hiking_note"].sum()
)

Original candidate paths : 559
Excluded paths           : 2
Retained candidate paths : 557

Exclusion reasons:
foot=no: 0
explicit non-hiking note: 2


### Candidate Path Filtering Decision

The original dataset contained 559 OpenStreetMap candidate path geometries.

A conservative evidence-based filtering strategy was applied rather than
removing paths simply because optional OSM attributes were missing.

The assessment found:

- no path explicitly tagged with `foot=no`;
- two paths with pedestrian permit/private conditions;
- four paths with general access restrictions;
- two paths explicitly described as `Only for Mountainbikes`.

The two explicitly non-hiking paths were excluded from the hiking candidate
dataset.

General `access=no` records were not automatically excluded when pedestrian
access information indicated that walking was permitted or required a permit.

After filtering, 557 candidate paths remained.

These records are treated as retained hiking/path candidates and not as
independently verified official hiking trails.

In [13]:
clean_trails = retained_paths.copy()

# Reset the row index while preserving OSM identifiers.
clean_trails = clean_trails.reset_index(drop=True)

print("Clean candidate trail dataset created.")
print("Rows:", len(clean_trails))
print("CRS:", clean_trails.crs)

print("\nHighway distribution:")
print(clean_trails["highway"].value_counts())

print("\nGeometry distribution:")
print(clean_trails.geometry.geom_type.value_counts())

Clean candidate trail dataset created.
Rows: 557
CRS: EPSG:4326

Highway distribution:
highway
track        289
footway      124
path         123
steps         20
bridleway      1
Name: count, dtype: int64

Geometry distribution:
LineString    557
Name: count, dtype: int64


In [14]:
print("ID QUALITY CHECK")
print("=" * 60)

print("Rows:", len(clean_trails))
print("Missing id:", clean_trails["id"].isna().sum())
print("Duplicate id:", clean_trails["id"].duplicated().sum())

print("\nExample IDs:")
print(clean_trails["id"].head(10).tolist())

print("\nOSM ID quality:")
print("Missing osm_id:", clean_trails["osm_id"].isna().sum())
print("Duplicate osm_id:", clean_trails["osm_id"].duplicated().sum())

ID QUALITY CHECK
Rows: 557
Missing id: 0
Duplicate id: 0

Example IDs:
['way/100146710', 'way/100146711', 'way/100148512', 'way/100398799', 'way/100398800', 'way/100398801', 'way/100399678', 'way/100399681', 'way/100399683', 'way/100399685']

OSM ID quality:
Missing osm_id: 0
Duplicate osm_id: 0


In [15]:
elevation_summary_file = (
    PROCESSED_DIR
    / "elevation"
    / "knuckles_trail_summary.csv"
)

print(
    "Elevation summary exists:",
    elevation_summary_file.exists()
)

if not elevation_summary_file.exists():
    raise FileNotFoundError(
        f"Elevation summary not found: {elevation_summary_file}"
    )

elevation_summary = pd.read_csv(
    elevation_summary_file
)

print("\nElevation summary loaded.")
print("Rows:", len(elevation_summary))
print("Columns:", len(elevation_summary.columns))

print("\nColumns:")
print(elevation_summary.columns.tolist())

elevation_summary.head()

Elevation summary exists: True

Elevation summary loaded.
Rows: 559
Columns: 8

Columns:
['trail_id', 'minimum_elevation_m', 'maximum_elevation_m', 'elevation_gain_m', 'elevation_loss_m', 'profile_complete', 'sample_spacing_max_m', 'direction']


,trail_id,minimum_elevation_m,maximum_elevation_m,elevation_gain_m,elevation_loss_m,profile_complete,sample_spacing_max_m,direction
0,way/100146710,1275.355835,1329.643188,137.471680,83.390137,True,60,OSM way node order
1,way/100146711,1024.330322,1281.052856,485.684814,287.749878,True,60,OSM way node order
2,way/100146901,1299.942383,1402.415527,6.243042,108.373413,True,60,OSM way node order
3,way/100148512,1332.734375,1420.415405,116.777222,29.096191,True,60,OSM way node order
4,way/100149325,1077.589600,1165.754395,0.000000,88.164795,True,60,OSM way node order


In [16]:
print("ELEVATION ID QUALITY CHECK")
print("=" * 60)

print(
    "Rows:",
    len(elevation_summary)
)

print(
    "Missing trail_id:",
    elevation_summary["trail_id"].isna().sum()
)

print(
    "Duplicate trail_id:",
    elevation_summary["trail_id"].duplicated().sum()
)

print("\nExample trail IDs:")

print(
    elevation_summary["trail_id"]
    .head(10)
    .tolist()
)

ELEVATION ID QUALITY CHECK
Rows: 559
Missing trail_id: 0
Duplicate trail_id: 0

Example trail IDs:
['way/100146710', 'way/100146711', 'way/100146901', 'way/100148512', 'way/100149325', 'way/100398799', 'way/100398800', 'way/100398801', 'way/100399678', 'way/100399681']
